# Classification via LLM génératif, avec exemples (in-context learning)

À rendre (mail avec objet MIDS TP5 NOM PRENOM) pour le mercredi 8 avril)

In [ ]:
!pip install -q transformers datasets

from datasets import load_dataset
import random
from sklearn.metrics import accuracy_score
from tqdm import tqdm


/home/fatoumata/Documents/NLP/TP5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Générer du texte à partir de prompts

In [2]:
# source https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto" # permet d'optimiser l'affectation des processeurs
)




ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Formatage des prompts

In [4]:
# exemples de prompts utilisant des "rôles"
# voir https://huggingface.co/docs/transformers/chat_templating
messages = [
  [{"role": "system",
    "content": "You are an experienced psychologist and you are very helpful to patients."},
  {"role": "user", "content": "I am suffering from insomnia. Could you help me?"}
  ],
  [
  {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
  {"role": "user", "content": "Give me a short introduction to large language models."}
  ]]

texts = [ tokenizer.apply_chat_template(message,
                                        tokenize=False,
                                        add_generation_prompt=True) for message in messages]
for t in texts:
  print(t)



<|im_start|>system
You are an experienced psychologist and you are very helpful to patients.<|im_end|>
<|im_start|>user
I am suffering from insomnia. Could you help me?<|im_end|>
<|im_start|>assistant

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Give me a short introduction to large language models.<|im_end|>
<|im_start|>assistant



## Tokenization

In [5]:
model_inputs = tokenizer(texts, return_tensors="pt", padding=True, padding_side="left").to(model.device)

print(model_inputs)



{'input_ids': tensor([[151643, 151643, 151644,   8948,    198,   2610,    525,    458,  10321,
          53495,    323,    498,    525,   1602,  10950,    311,   6835,     13,
         151645,    198, 151644,    872,    198,     40,   1079,  15691,    504,
          82411,     13,  16503,    498,   1492,    752,     30, 151645,    198,
         151644,  77091,    198],
        [151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,  35127,    752,    264,
           2805,  16800,    311,   3460,   4128,   4119,     13, 151645,    198,
         151644,  77091,    198]], device='cuda:0'), 'attention_mask': tensor([[0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1

## Génération

In [6]:
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=128
)

# generated_ids contient en fait les prompts
# on les coupe ici pour n'avoir que les réponses
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)


In [7]:
for r in response:
  print(r)

Certainly! Insomnia can be a challenging condition, but there are several strategies that might help improve your sleep quality. Here are some steps you can take:

1. **Establish a Routine**: Go to bed and wake up at the same time every day, even on weekends. This helps regulate your body's internal clock.

2. **Create a Relaxing Bedtime Ritual**: Engage in activities that relax you before bedtime, such as reading or listening to calming music.

3. **Limit Exposure to Screens Before Bed**: The blue light emitted by screens (phones, tablets, computers) can interfere with your ability to fall asleep. Try to
Large Language Models (LLMs) are artificial intelligence systems that have been trained on vast amounts of text data from the internet and other sources. They use advanced algorithms to understand natural language and generate human-like responses to questions or prompts.

The development of LLMs has led to significant advancements in fields such as natural language processing, machin

# Application : classification de documents

Il s'agit d'utiliser un LLM génératif pour classer des documents,
en fournissant quelques exemples dans le prompt
(= "in-context learning", ou "few-shot prompting" ...)

## Dataset ag_news : news classées en 4 thèmes

In [8]:
# AG news dataset
# Labels: 0=World, 1=Sports, 2=Business, 3=Sci/Tech
# https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset
dataset = load_dataset("ag_news")

# on réduit ici le dataset
# NB: le "train" sera utilisé pour piocher des exemple pour le in-context learning
train_data = dataset['train'].shuffle(seed=42).select(range(200))
test_data = dataset['test'].shuffle(seed=42).select(range(100))

label_names = ["World", "Sports", "Business", "Sci/Tech"]


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [10]:
print(test_data[0].keys())
print(test_data[0]['text'])
print(test_data[0]['label'])

dict_keys(['text', 'label'])
Indian board plans own telecast of Australia series The Indian cricket board said on Wednesday it was making arrangements on its own to broadcast next month #39;s test series against Australia, which is under threat because of a raging TV rights dispute.
1


## TODO: classifieur de type LLM génératif avec in-context learning
- méthode de construction d'un prompt avec exemples
  * prompt structuré en
    - role "system" : indications générales. Par exemple "You are an expert in news. Your task is to classify news into the four following categories : ... etc..."
    - role "user" : input de l'utilisateur
    - role "assistant" : réponse pour un input
  * un exemple fourni dans le prompt instancie les role "user" et "assistant"
  * faites en sorte d'équilibrer les classes dans les exemples fournis, et que toutes les classes soient représentées.

- génération

- méthode de récupération de la catégorie, au sein du texte généré
  * limiter le texte généré
  * plusieurs stratégies possibles : première cat trouvée dans la réponse par exemple
  * quantifier les cas où aucune réponse valide n'est trouvée

Vous comparerez les performances pour plusieurs valeurs de k (par ex. k=0, k=4, k=8)
(attention pour le même sous-ensemble de test, cf. random supra).

Eventuellement: testez si modifier la formulation du prompt a un impact.

## Implémentation du classifieur avec few-shot prompting

In [ ]:
import random

def select_examples(train_data, k):
    if k == 0:
        return []
    examples_per_class = k // 4
    extra = k % 4
    examples = []
    for label in range(4):
        class_examples = [ex for ex in train_data if ex['label'] == label]
        num_to_select = examples_per_class + (1 if extra > 0 else 0)
        selected = random.sample(class_examples, min(num_to_select, len(class_examples)))
        examples.extend(selected)
        if extra > 0:
            extra -= 1
    random.shuffle(examples)
    return examples

def build_prompt(examples, test_text):
    system_content = "You are an expert in news classification. Your task is to classify news articles into one of the four categories: World, Sports, Business, Sci/Tech. Respond with only the category name."
    messages = [{"role": "system", "content": system_content}]
    for ex in examples:
        messages.append({"role": "user", "content": f"Classify this news: {ex['text']}"})
        messages.append({"role": "assistant", "content": label_names[ex['label']]})
    messages.append({"role": "user", "content": f"Classify this news: {test_text}"})
    return messages

def classify_text(text, examples):
    prompt = build_prompt(examples, text)
    text_prompt = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text_prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    generated = outputs[0][len(inputs['input_ids'][0]):]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip()
    # Parse response
    response_lower = response.lower()
    for i, label in enumerate(label_names):
        if label.lower() in response_lower:
            return i
    return -1  # invalid

# Evaluation
results = {}
for k in [0, 4, 8]:
    examples = select_examples(list(train_data), k)  # convert to list
    preds = []
    invalids = 0
    for item in tqdm(test_data):
        pred = classify_text(item['text'], examples)
        if pred == -1:
            invalids += 1
            pred = 0  # default to 0 or something, but for accuracy, perhaps skip or count as wrong
        preds.append(pred)
    acc = accuracy_score([item['label'] for item in test_data], preds)
    results[k] = {'accuracy': acc, 'invalids': invalids}
    print(f"k={k}, accuracy: {acc:.3f}, invalids: {invalids}")

# Optional: test different prompt formulations
# For example, change system_content to "Classify the following news article into: World, Sports, Business, Sci/Tech. Answer only with the category."

In [ ]:
# Test different prompt formulations
system_prompts = [
    "You are an expert in news classification. Your task is to classify news articles into one of the four categories: World, Sports, Business, Sci/Tech. Respond with only the category name.",
    "Classify the following news article into: World, Sports, Business, Sci/Tech. Answer only with the category.",
    "As a news classifier, categorize this article into one of: World, Sports, Business, Sci/Tech. Output just the category."
]

def build_prompt_v2(examples, test_text, system_content):
    messages = [{"role": "system", "content": system_content}]
    for ex in examples:
        messages.append({"role": "user", "content": f"Classify this news: {ex['text']}"})
        messages.append({"role": "assistant", "content": label_names[ex['label']]})
    messages.append({"role": "user", "content": f"Classify this news: {test_text}"})
    return messages

def classify_text_v2(text, examples, system_content):
    prompt = build_prompt_v2(examples, text, system_content)
    text_prompt = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text_prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    generated = outputs[0][len(inputs['input_ids'][0]):]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip()
    # Parse response
    response_lower = response.lower()
    for i, label in enumerate(label_names):
        if label.lower() in response_lower:
            return i
    return -1  # invalid

# Evaluation with different prompts for k=4
k = 4
examples = select_examples(list(train_data), k)
for i, sys_prompt in enumerate(system_prompts):
    preds = []
    invalids = 0
    for item in tqdm(test_data, desc=f"Prompt {i+1}"):
        pred = classify_text_v2(item['text'], examples, sys_prompt)
        if pred == -1:
            invalids += 1
            pred = 0  # default
        preds.append(pred)
    acc = accuracy_score([item['label'] for item in test_data], preds)
    print(f"Prompt {i+1}: accuracy: {acc:.3f}, invalids: {invalids}")
    print(f"System: {sys_prompt[:50]}...")